In [ ]:
### Annotating Different Clusters

In [ ]:
library(dplyr)
library(Matrix)
library(data.table)
library(Seurat)
library(ggplot2)
library(RColorBrewer)
library(cowplot)
library(future)
library(viridis)
library(SeuratObject)
library(tidyr)

In [ ]:
seuobj = readRDS("/data/Bin200.merge.debatch.rds")

In [ ]:
### read anno info
bayes.anno = read.csv("/data/layer.anno.csv")
bayes.anno[1:2,]

In [ ]:
### add layer info
for (i in c("16")){
    seuobj@meta.data[,paste0("Bayes.anno",i)] <- NA
    anno.i = bayes.anno %>% filter(res %in% paste0("Bayes",i))
    for (clu in anno.i$Clu){
        clu.anno = anno.i %>% filter(Clu %in% clu)
        clu.anno = clu.anno[,"anno"]
        seuobj@meta.data[which(seuobj@meta.data[,paste0("Bayes",i)] == clu),paste0("Bayes.anno",i)] = clu.anno
        }
    }

In [ ]:
cols = c("#ed0c7f","#377eb9","#4dae48","#974f9f","#fed700","#f57e1f","#a65628",    "#66c2a5","#fdbf6f","#e31a1c","#b3b3b3","#bdbdbd",
        "#a6cee3","#e78ac3","#b2df8a","#e5c494")
names(cols) = c("L1","L2","L3","L4","L5","L6","WM","L4/5","L6-WM","Blood cell","Del","undetermined",
                "L23","9","15","17")

In [ ]:
### plot anno umap
pdf("/data/Bin200.anno.UMAP.pdf",width=10)
for (i in  c("16")){
    p = DimPlot(seuobj,reduction="stereo",group.by = paste0("Bayes.anno",i),label = FALSE,label.size = 0.6,pt.size = 0.001,raster=FALSE,cols = cols)+coord_fixed()+ theme_void() +
          theme(axis.text = element_blank(), axis.ticks = element_blank(),axis.title = element_blank(),panel.background = element_rect(fill = "black"))
    print(p)
    }
dev.off()

In [ ]:
############## split cluster umap
options(repr.plot.width=15, repr.plot.height=5)  
pdf("/data/Bin200.anno.split.UMAP.pdf")
for (i in c("20")){
    p <-  ggplot(seuobj@meta.data,aes(as.numeric(coor_x), as.numeric(coor_y), fill= seuobj@meta.data[,paste0("Bayes.anno",i)]))+geom_tile() + theme_classic()+ coord_fixed()+ scale_fill_manual(values =cols)+#theme(panel.background = element_rect(fill = 'gray0', colour = 'gray0'))+
            labs(fill=seuobj@meta.data[,paste0("Bayes.anno",i)],x = "coor_x",y = "coor_y")+facet_wrap(. ~ seuobj@meta.data[,paste0("Bayes.anno",i)],ncol = 3)+guides(fill=guide_legend(title=paste0("Bayes.anno",i)))
    print(p)
    }
dev.off()

In [ ]:
### plot layer marker
features = c("RELN","FABP7",  ##l1
             "C1QL2","LAMP5","VIP", ##l2
             "COL5A2", ##l3
             "TSHZ2","RORB", ##l4
             "HTR2A","HS3ST2","PCP4","HTR2C",  ##l5
             "NTNG2","THEMIS","SYT6","SEMA3E","CCN2","KRT17",  ##l6
             "MBP")

In [ ]:
pdf("/data/layerMarker.featureplot.pdf",height=3,width=9)
for (i in features){
        allmean = data.frame(seuobj$coor_x,seuobj$coor_y,seuobj@assays$Spatial@data[i,])
        colnames(allmean) <- c("coor_x","coor_y","Exp")
        options(repr.plot.width=10, repr.plot.height=6) 
        p = ggplot()+geom_point(data=allmean,aes(x=coor_x,y=coor_y,color=Exp),size=0.3, alpha=1.3,stroke = 0)+ scale_colour_gradientn(values = seq(0,1,0.2),colours = c('black','blue','green','yellow','orange','red'))+
            theme_classic()+coord_fixed()+theme(panel.background = element_rect(fill="black"))+#labs(title = paste0(j,"_",gene))+ggtitle(paste0(j,"_",i))+
            theme(panel.grid.major = element_blank(), panel.grid.minor = element_blank(),panel.border = element_blank(),axis.line = element_blank(),
                axis.text.x = element_blank(), axis.text.y = element_blank(),axis.title.x = element_blank(),axis.title.y = element_blank(),axis.ticks.x =element_blank(),axis.ticks.y =element_blank())+ggtitle(i) ##去掉xy坐标
        print(p)
    }
dev.off()